In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Paths ----------
PROJECT = Path("/Users/ashokdas/envs363_563/geo-demo")
SRC = PROJECT / "data" / "PRT_SHODIST_all.csv"
OUT = PROJECT / "outputs"
OUT.mkdir(exist_ok=True)

print("Input exists:", SRC.exists())
print("Output folder:", OUT)

# ---------- Settings ----------
META = ["name", "Class", "Desig_APA", "Municip"]
MIN_OBS, MIN_SPAN = 10, 5
STABLE = 0.5

# ---------- 1. Identify date columns ----------
header = pd.read_csv(SRC, nrows=0).columns.tolist()

date_cols, dates = [], []
for c in header:
    if c in META:
        continue
    try:
        dates.append(pd.Timestamp(c))
        date_cols.append(c)
    except (ValueError, TypeError):
        pass

print(len(date_cols), "date columns")
print(min(dates), "to", max(dates))

# ---------- 2. Sample check ----------
sample = pd.read_csv(SRC, nrows=500, usecols=META + date_cols, low_memory=False)
vals = sample[date_cols].apply(pd.to_numeric, errors="coerce")
print(vals.notna().sum().sum(), "valid cells in 500 transects")
print(vals.stack().describe())

# ---------- 3. Rate per transect (takes some time ) ----------
ref = min(dates)
t_years = np.array([(d - ref).days / 365.25 for d in dates])

rows = []
for chunk in pd.read_csv(SRC, chunksize=500, usecols=META + date_cols, low_memory=False):
    v = chunk[date_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    for i in range(v.shape[0]):
        m = ~np.isnan(v[i])
        n = int(m.sum())
        meta = chunk.iloc[i][META].to_dict()
        if n < MIN_OBS:
            rows.append({**meta, "n_obs": n, "rate": np.nan, "status": "insufficient"})
            continue
        span = t_years[m].max() - t_years[m].min()
        if span < MIN_SPAN:
            rows.append({**meta, "n_obs": n, "rate": np.nan, "status": "span_short"})
            continue
        slope = np.polyfit(t_years[m], v[i, m], 1)[0]
        rows.append({**meta, "n_obs": n, "rate": slope, "status": "ok"})

rates = pd.DataFrame(rows)
print(rates["status"].value_counts())

# ---------- 4. Classify trends ----------
ok = rates["status"] == "ok"
rates.loc[ok, "trend"] = np.select(
    [rates.loc[ok, "rate"] < -STABLE, rates.loc[ok, "rate"] > STABLE],
    ["erosion", "accretion"], default="stable")
print(rates["trend"].value_counts())
print("Median rate (m/yr):", rates.loc[ok, "rate"].median())

# ---------- 5. Coverage by year ( wait to load ) ----------
valid_per_year = {}
for chunk in pd.read_csv(SRC, chunksize=500, usecols=date_cols, low_memory=False):
    v = chunk.apply(pd.to_numeric, errors="coerce").notna().sum()
    for col, n in v.items():
        y = pd.Timestamp(col).year
        valid_per_year[y] = valid_per_year.get(y, 0) + n

coverage = pd.Series(valid_per_year, name="valid_measurements").sort_index()
print(coverage.head(), coverage.tail())

# ---------- 6. Save outputs ----------
rates.to_csv(OUT / "transect_rates.csv", index=False)
coverage.to_csv(OUT / "coverage_by_year.csv")
print("Saved:", (OUT / "transect_rates.csv").resolve())

# ---------- 7. Chart 1: rate distribution ----------
r = rates.loc[ok, "rate"].clip(-10, 10)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(r, bins=60, color="#3b6e8f", edgecolor="white")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Change rate (m/yr), clipped to ±10")
ax.set_ylabel("Transects")
ax.set_title("Shoreline change rates (demo)")
plt.tight_layout()
plt.savefig(OUT / "rate_distribution.png", dpi=150)
plt.show()

# ---------- 8. Chart 2: coverage over time ----------
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(coverage.index, coverage.values, marker="o")
ax.set_xlabel("Year")
ax.set_ylabel("Valid measurements")
ax.set_title("Data coverage over time")
plt.tight_layout()
plt.savefig(OUT / "coverage_by_year.png", dpi=150)
plt.show()